# IchiV2 Live vs Backtest — GMX Vault (ichiv2_gmx_vault)


## Setup

In [ ]:
import nest_asyncio
nest_asyncio.apply()
import os
from pathlib import Path

def find_project_root():
    """
    Find the project root by searching for configs directory.
    """
    current = Path.cwd()

    # If already at project root
    if (current / "configs").exists():
        return current

    # Search parent directories
    for parent in current.parents:
        if (parent / "configs").exists():
            return parent

    # Fallback: return current directory
    print("Warning: Could not find project root. Using current directory.")
    return current

PROJECT_ROOT = find_project_root()
print(f"Working directory: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
repo_root = PROJECT_ROOT

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from copy import deepcopy
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import sqlite3

import sys
from pathlib import Path

# Apply GMX monkeypatch BEFORE importing freqtrade
_gmx_defi = Path(repo_root) / 'deps' / 'web3-ethereum-defi'
if _gmx_defi.exists():
    sys.path.insert(0, str(_gmx_defi))
    print(f'\u2713 web3-ethereum-defi path: {_gmx_defi}')

from eth_defi.gmx.freqtrade.monkeypatch import patch_freqtrade
patch_freqtrade()
print('\u2713 GMX monkeypatch applied')

# Register custom pairlist plugins so schema validation accepts them
from freqtrade.constants import AVAILABLE_PAIRLISTS
for name in ("HistoricalVolumePairList", "GMXLiquidityFilter"):
    if name not in AVAILABLE_PAIRLISTS:
        AVAILABLE_PAIRLISTS.append(name)
print(f'\u2713 Custom pairlists registered: {[n for n in ("HistoricalVolumePairList", "GMXLiquidityFilter") if n in AVAILABLE_PAIRLISTS]}')

# Prefer repo-bundled Freqtrade
_ft_local = Path(repo_root) / 'freqtrade-develop'
if _ft_local.exists():
    sys.path.insert(0, str(_ft_local))
    print(f'\u2713 Using repo freqtrade: {_ft_local}')
else:
    print(f'Warning: Repo freqtrade not found at {_ft_local} (using installed freqtrade)')

import freqtrade
print(f'\u2713 Freqtrade loaded from: {Path(freqtrade.__file__).resolve()}')

from freqtrade.configuration import Configuration
from freqtrade.resolvers import StrategyResolver
from freqtrade.data.history import get_timerange
from freqtrade.plot.plotting import generate_candlestick_graph
from freqtrade.persistence import Trade, init_db
from freqtrade.optimize.backtesting import Backtesting
from freqtrade.data.metrics import (
    calculate_max_drawdown,
    calculate_sharpe,
    calculate_sortino,
    calculate_calmar,
    calculate_expectancy
)


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
from pathlib import Path

# Get to the repo root
os.chdir(PROJECT_ROOT)

# Verify we're in the right place
print(f"Current directory: {os.getcwd()}")
print(f"Configs directory exists: {os.path.exists('configs')}")
print(f"Config file exists: {os.path.exists('configs/ichiv2_gmx.json')}")
print(f"Secrets file exists: {os.path.exists('configs/ichiv2_gmx.secrets.json')}")
print(f"Data directory exists: {os.path.exists('user_data/data/gmx/futures')}")

## Configuration

**Customize these settings for your backtest:**

In [ ]:
# ===== BOT MODE =====
# "static"  -> ichiv2_gmx (fixed 41 pairs, StaticPairList)
# "vault"   -> ichiv2_gmx_vault (174-pair universe, HistoricalVolumePairList top 40)
BOT_MODE = "vault"  # <-- CHANGE THIS

# ===== MODE-DEPENDENT SETTINGS =====
if BOT_MODE == "static":
    CONFIG_FILES = ["configs/ichiv2_gmx.json", "configs/ichiv2_gmx.secrets.json"]
    LIVE_DB_PATH = "db/archive/ichiv2_gmx/ichiv2_gmx_2026-02-06_2026-03-19.sqlite.backup"
    TIMERANGE = "20260206-20260319"
    STARTING_BALANCE = 100  # dry_run_wallet from config
    LIVE_API_BASE = os.environ.get("FT_API_BASE", "http://ichiv2-gmx.tradingstrategy.ai")
elif BOT_MODE == "vault":
    CONFIG_FILES = ["configs/ichiv2_gmx_prod_vault.json", "configs/ichiv2_gmx_prod_vault.secrets.json"]
    LIVE_DB_PATH = "db/archive/ichiv2_gmx_vault/ichiv2_gmx_vault_2026-03-15_2026-03-19.sqlite.backup"
    TIMERANGE = "20260101-20260319"  # Start early for indicator warmup
    STARTING_BALANCE = 100  # adjust to match vault starting balance
    LIVE_API_BASE = os.environ.get("FT_API_BASE", "http://ichiv2-gmx-vault.tradingstrategy.ai")
else:
    raise ValueError(f"Unknown BOT_MODE: {BOT_MODE}")

# ===== COMMON SETTINGS =====
STRATEGY = "IchiV2_LS_Static"
EXCHANGE = "gmx"
TIMEFRAME = "1h"
STAKE_CURRENCY = "USDC"
CUSTOM_PAIRS = None  # Set to list to override, or None to use config

# ===== LIVE DATA CONFIGURATION =====
COMPARE_WITH_LIVE = True

# Choose how live data is sourced:
# - "api": Pull from running bot (requires bot to be up)
# - "db":  Read sqlite trades directly (works with archived backups)
LIVE_SOURCE = "db"  # Use archived DB by default

# Live API settings (only used if LIVE_SOURCE == "api")
LIVE_API_USER = os.environ.get("FT_API_USER", "ft2admin")
LIVE_API_PASS = os.environ.get("FT_API_PASS", "")
LIVE_API_TIMEOUT_SECS = 20

# Corruption filter for vault DB (some trades have wei-scale profit values)
PROFIT_ABS_MAX = 10000  # Exclude trades with abs(profit) > this threshold

print(f"BOT_MODE: {BOT_MODE}")
print(f"Config: {CONFIG_FILES[0]}")
print(f"Live DB: {LIVE_DB_PATH}")
print(f"Timerange: {TIMERANGE}")


In [ ]:
# Load configuration with in-memory overrides for backtesting

config = Configuration.from_files(CONFIG_FILES)
config["strategy"] = STRATEGY
config["timeframe"] = TIMEFRAME
config["stake_currency"] = STAKE_CURRENCY
config["datadir"] = Path("user_data/data/gmx")
config["dry_run"] = True
config["dry_run_wallet"] = STARTING_BALANCE
config["timerange"] = TIMERANGE
config["max_open_trades"] = config.get("max_open_trades", 10)

# For vault mode, ensure dynamic pairlist is enabled
if BOT_MODE == "vault":
    config["enable_dynamic_pairlist"] = True
    print(f"\n\u2713 Dynamic pairlist ENABLED (HistoricalVolumePairList)")
    pairlist_methods = [p.get("method") for p in config.get("pairlists", [])]
    print(f"  Pairlist chain: {pairlist_methods}")

if CUSTOM_PAIRS:
    config["pairs"] = CUSTOM_PAIRS

print(f"\n\u2713 Configuration loaded")
print(f"  Strategy: {STRATEGY}")
print(f"  Exchange: {EXCHANGE}")
print(f"  Timeframe: {TIMEFRAME}")
print(f"  Timerange: {TIMERANGE}")
print(f"  Data dir: {config['datadir']}")
print(f"  Max Open Trades: {config['max_open_trades']}")
print(f"  Starting Balance: {STARTING_BALANCE} {STAKE_CURRENCY}")
print(f"  Pairs: {len(config.get('exchange', {}).get('pair_whitelist', []))} in whitelist")


In [ ]:
# Optional: View/manage market cap cache
import json
from pathlib import Path
from datetime import datetime, timedelta

cache_file = Path("user_data/strategies/market_cap_cache.json")

if cache_file.exists():
    with open(cache_file, 'r') as f:
        cache = json.load(f)
    
    cache_time = datetime.fromisoformat(cache['timestamp'])
    age = datetime.now() - cache_time
    is_valid = age < timedelta(hours=72)
    
    print(f"Market Cap Cache Status:")
    print(f"  Last updated: {cache_time}")
    print(f"  Age: {age}")
    print(f"  Valid: {'Yes' if is_valid else 'No (will refresh on next API call)'}")
    print(f"  Coins cached: {len(cache['marketcap'])}")
    print(f"\n  Sample market caps (in billions USD):")
    for symbol, mcap in list(cache['marketcap'].items())[:5]:
        print(f"    {symbol}: ${mcap/1e9:.2f}B")
else:
    print("No cache file found - will fetch from CoinGecko on first run")

## Run Backtest

In [ ]:
print("Starting backtest...\n")

backtesting = Backtesting(config)
backtesting._set_strategy(backtesting.strategylist[0])
strategy = backtesting.strategy

print("Loading data...")
data, timerange = backtesting.load_bt_data()

print("Processing indicators...")
processed = strategy.advise_all_indicators(data)
min_date, max_date = get_timerange(processed)

print(f"\nBacktest period: {min_date} to {max_date}")
print(f"Running backtest on {len(processed)} pairs...\n")

Trade.reset_trades()
result = backtesting.backtest(
    processed=deepcopy(processed),
    start_date=min_date,
    end_date=max_date,
)

backtest_trades = result["results"].copy()

#FILTER OUT FORCE_EXIT TRADES FIRST (before any calculations)
# Uncomment below to filter out force_exit trades
# original_count = len(backtest_trades)
# backtest_trades = backtest_trades[backtest_trades['exit_reason'] != 'force_exit'].copy()
# filtered_count = original_count - len(backtest_trades)
# if filtered_count > 0:
#     print(f"\nFiltered out {filtered_count} force_exit trade(s) from backtest")
#     print(f"   Remaining backtest trades: {len(backtest_trades)}")

backtest_trades['profit_ratio_pct'] = backtest_trades['profit_ratio'] * 100
backtest_trades['duration_hours'] = backtest_trades['trade_duration'] / 60.0
backtest_trades['duration_days'] = backtest_trades['duration_hours'] / 24.0

Backtesting.cleanup()
Trade.reset_trades()

print(f"\n\u2713 Backtest complete: {len(backtest_trades)} trades")

## Load Live Data (API preferred)

In [ ]:
live_trades = None
live_profit = None  # /api/v1/profit response
live_balance = None  # /api/v1/balance response (starting_capital denominator)
live_open_trades = None  # /api/v1/status response (open trades)


def _ft_api_url(path: str) -> str:
    base = (LIVE_API_BASE or "").rstrip("/")
    if not path.startswith("/"):
        path = "/" + path
    return f"{base}{path}"


def _ft_api_basic_headers() -> dict:
    import base64

    if not LIVE_API_USER or not LIVE_API_PASS:
        raise ValueError(
            "Live API credentials missing. Set env vars FT_API_USER / FT_API_PASS (recommended)."
        )
    token = base64.b64encode(f"{LIVE_API_USER}:{LIVE_API_PASS}".encode()).decode()
    return {
        "Authorization": f"Basic {token}",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }


def _ft_api_get_json(path: str):
    import json
    import urllib.error
    import urllib.request

    req = urllib.request.Request(_ft_api_url(path), headers=_ft_api_basic_headers())
    try:
        with urllib.request.urlopen(req, timeout=LIVE_API_TIMEOUT_SECS) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Live API HTTP {e.code} for {path}: {body}")


def _parse_dt(val):
    """Parse timestamps coming from Freqtrade JSON."""
    if val is None:
        return pd.NaT
    try:
        return pd.to_datetime(val, utc=True)
    except Exception:
        return pd.to_datetime(val, unit="ms", utc=True)


def _normalize_closed_trade_row(t: dict) -> dict:
    open_dt = _parse_dt(t.get("open_date") or t.get("open_date_utc") or t.get("open_timestamp"))
    close_dt = _parse_dt(t.get("close_date") or t.get("close_date_utc") or t.get("close_timestamp"))

    profit_abs = (
        t.get("close_profit_abs")
        if t.get("close_profit_abs") is not None
        else t.get("profit_abs")
    )
    profit_ratio = (
        t.get("close_profit")
        if t.get("close_profit") is not None
        else t.get("profit_ratio")
    )

    trade_id = t.get("trade_id") if t.get("trade_id") is not None else t.get("id")

    return {
        "trade_id": trade_id,
        "pair": t.get("pair"),
        "stake_amount": t.get("stake_amount"),
        "amount": t.get("amount"),
        "open_date": open_dt,
        "close_date": close_dt,
        "open_rate": t.get("open_rate"),
        "close_rate": t.get("close_rate"),
        "profit_abs": float(profit_abs or 0.0),
        "profit_ratio": float(profit_ratio or 0.0),
        "exit_reason": t.get("exit_reason") or t.get("sell_reason"),
        "is_short": t.get("is_short"),
        "leverage": t.get("leverage"),
        "is_open": False,
    }


def _normalize_open_trade_row(t: dict, close_dt: "pd.Timestamp") -> dict:
    open_dt = _parse_dt(t.get("open_date") or t.get("open_timestamp"))

    profit_abs = t.get("total_profit_abs")
    if profit_abs is None:
        profit_abs = t.get("profit_abs")

    profit_ratio = t.get("total_profit_ratio")
    if profit_ratio is None:
        profit_ratio = t.get("profit_ratio")

    current_rate = t.get("current_rate")

    trade_id = t.get("trade_id") if t.get("trade_id") is not None else t.get("id")

    return {
        "trade_id": trade_id,
        "pair": t.get("pair"),
        "stake_amount": t.get("stake_amount"),
        "amount": t.get("amount"),
        "open_date": open_dt,
        "close_date": close_dt,
        "open_rate": t.get("open_rate"),
        "close_rate": current_rate,
        "profit_abs": float(profit_abs or 0.0),
        "profit_ratio": float(profit_ratio or 0.0),
        "exit_reason": "open_mtm",
        "is_short": t.get("is_short"),
        "leverage": t.get("leverage"),
        "is_open": True,
    }


if COMPARE_WITH_LIVE and str(LIVE_SOURCE).lower() == "api":
    try:
        # Public endpoint - quick connectivity check
        ping = _ft_api_get_json("/api/v1/ping")
        print(f"\u2713 Live API ping: {ping}")

        # Truth metrics (includes mark-to-market for open trades)
        live_profit = _ft_api_get_json("/api/v1/profit")

        # Starting balance denominator (used by /profit)
        live_balance = _ft_api_get_json("/api/v1/balance")

        # Closed trades list
        trades_payload = _ft_api_get_json("/api/v1/trades?limit=500&offset=0")
        closed_trades = trades_payload.get("trades", []) if isinstance(trades_payload, dict) else trades_payload

        # Open trades (status)
        live_open_trades = _ft_api_get_json("/api/v1/status")
        open_trades = live_open_trades if isinstance(live_open_trades, list) else []

        close_dt = pd.Timestamp.utcnow()

        rows = [_normalize_closed_trade_row(t) for t in closed_trades]
        rows += [_normalize_open_trade_row(t, close_dt=close_dt) for t in open_trades]
        live_trades = pd.DataFrame(rows)

        if len(live_trades) > 0:
            live_trades["open_date"] = pd.to_datetime(live_trades["open_date"], utc=True).dt.tz_convert(None)
            live_trades["close_date"] = pd.to_datetime(live_trades["close_date"], utc=True).dt.tz_convert(None)

            live_trades["trade_duration"] = (
                (live_trades["close_date"] - live_trades["open_date"]).dt.total_seconds() / 60.0
            )
            live_trades["profit_ratio_pct"] = live_trades["profit_ratio"] * 100
            live_trades["duration_hours"] = live_trades["trade_duration"] / 60.0
            live_trades["duration_days"] = live_trades["duration_hours"] / 24.0

        print("\n\u2713 Live API profit (Telegram-style):")
        print(
            "  ROI: Closed trades\n"
            f"  {live_profit.get('profit_closed_coin'):.3f} {STAKE_CURRENCY} "
            f"({live_profit.get('profit_closed_percent_mean'):.2f}%) "
            f"({live_profit.get('profit_closed_percent'):.2f} Sigma%)"
        )
        print(
            "  ROI: All trades\n"
            f"  {live_profit.get('profit_all_coin'):.3f} {STAKE_CURRENCY} "
            f"({live_profit.get('profit_all_percent_mean'):.2f}%) "
            f"({live_profit.get('profit_all_percent'):.2f} Sigma%)"
        )
        print(f"  Total Trade Count: {live_profit.get('trade_count')} (closed={live_profit.get('closed_trade_count')})")

        print(f"\n\u2713 Loaded {len(live_trades)} live trades via API (closed + open_mtm)")

        # Optional sanity check: compare /trades (API) vs sqlite DB closed trades
        if os.path.exists(LIVE_DB_PATH):
            try:
                conn = sqlite3.connect(LIVE_DB_PATH)
                db_closed = pd.read_sql_query(
                    """
                    SELECT id as trade_id, pair, open_date, close_date, close_profit_abs as profit_abs
                    FROM trades
                    WHERE is_open = 0
                    ORDER BY id
                    """,
                    conn,
                )
                conn.close()

                api_closed = pd.DataFrame([_normalize_closed_trade_row(t) for t in closed_trades])

                if len(db_closed) > 0:
                    db_closed["open_date"] = pd.to_datetime(db_closed["open_date"])
                    db_closed["close_date"] = pd.to_datetime(db_closed["close_date"])

                print("\nAPI vs DB (closed trades) sanity check:")
                print(f"  API closed count: {len(api_closed)} | DB closed count: {len(db_closed)}")
                print(
                    f"  API closed profit_abs sum: {api_closed['profit_abs'].sum():.6f} | "
                    f"DB closed profit_abs sum: {db_closed['profit_abs'].sum():.6f}"
                )

                api_ids = set(api_closed["trade_id"].dropna().astype(int).tolist()) if "trade_id" in api_closed else set()
                db_ids = set(db_closed["trade_id"].dropna().astype(int).tolist())

                if api_ids and db_ids:
                    missing_in_api = sorted(list(db_ids - api_ids))
                    missing_in_db = sorted(list(api_ids - db_ids))
                    print(f"  Missing in API: {missing_in_api[:5]}" + (" ..." if len(missing_in_api) > 5 else ""))
                    print(f"  Missing in DB: {missing_in_db[:5]}" + (" ..." if len(missing_in_db) > 5 else ""))

            except Exception as e:
                print(f"  Warning: Sanity check failed: {e}")

    except Exception as e:
        print(f"Warning: Error loading live data from API: {e}")
        live_trades = None
        live_profit = None
        live_open_trades = None

elif COMPARE_WITH_LIVE and str(LIVE_SOURCE).lower() == "db":
    live_trades = None

    if os.path.exists(LIVE_DB_PATH):
        print(f"Loading live trades from: {LIVE_DB_PATH}\n")

        try:
            conn = sqlite3.connect(LIVE_DB_PATH)

            live_trades = pd.read_sql_query(
                """
                SELECT 
                    pair,
                    stake_amount,
                    amount,
                    open_date,
                    close_date,
                    open_rate,
                    close_rate,
                    close_profit_abs as profit_abs,
                    close_profit as profit_ratio,
                    exit_reason,
                    is_short,
                    leverage
                FROM trades
                ORDER BY close_date
                """,
                conn,
            )
            conn.close()

            live_trades["open_date"] = pd.to_datetime(live_trades["open_date"])
            live_trades["close_date"] = pd.to_datetime(live_trades["close_date"])

            live_trades["trade_duration"] = (
                (live_trades["close_date"] - live_trades["open_date"]).dt.total_seconds() / 60.0
            )

            live_trades["profit_ratio_pct"] = live_trades["profit_ratio"] * 100
            live_trades["duration_hours"] = live_trades["trade_duration"] / 60.0
            live_trades["duration_days"] = live_trades["duration_hours"] / 24.0

            # Filter out corrupted trades (wei-scale profit values in vault DB)
            if PROFIT_ABS_MAX is not None:
                corrupt_mask = live_trades["profit_abs"].abs() > PROFIT_ABS_MAX
                n_corrupt = corrupt_mask.sum()
                if n_corrupt > 0:
                    print(f"\n\u26a0 Filtering {n_corrupt} trades with |profit_abs| > {PROFIT_ABS_MAX} (corrupted)")
                    for _, row in live_trades[corrupt_mask].iterrows():
                        print(f"  - {row['pair']} profit_abs={row['profit_abs']:.2f}")
                    live_trades = live_trades[~corrupt_mask].reset_index(drop=True)

            print(f"\u2713 Loaded {len(live_trades)} live trades")
            print(f"  Period: {live_trades['open_date'].min()} to {live_trades['close_date'].max()}")
            print(f"  Total profit: {live_trades['profit_abs'].sum():.2f} {STAKE_CURRENCY}")

        except Exception as e:
            print(f"Warning: Error loading live database: {e}")
            live_trades = None

    else:
        print(f"Warning: Live database not found: {LIVE_DB_PATH}")

else:
    print("Live comparison disabled")


In [ ]:
if live_trades is not None and len(live_trades) > 0:
    live_trades.sort_values(by='close_date')
else:
    print("No live trades loaded (this is expected if the GMX bot is not running yet)")

## Helper Functions

In [ ]:
def calculate_performance_summary(trades_df, label="", starting_balance=1000):
    """Calculate comprehensive performance metrics for a trades DataFrame using Freqtrade methods."""
    if len(trades_df) == 0:
        return pd.DataFrame({'Metric': ['No trades'], 'Value': ['N/A']})
    
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['profit_ratio'] > 0])
    losing_trades = len(trades_df[trades_df['profit_ratio'] <= 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    
    total_profit_abs = trades_df['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100  
    avg_profit = trades_df['profit_ratio'].mean() * 100
    
    avg_win = trades_df[trades_df['profit_ratio'] > 0]['profit_ratio'].mean() * 100 if winning_trades > 0 else 0
    avg_loss = trades_df[trades_df['profit_ratio'] <= 0]['profit_ratio'].mean() * 100 if losing_trades > 0 else 0
    avg_duration = trades_df['duration_hours'].mean()
    
    min_date = trades_df['open_date'].min()
    max_date = trades_df['close_date'].max()
    
    drawdown = calculate_max_drawdown(
        trades_df,
        date_col='close_date',
        value_col='profit_abs',
        starting_balance=starting_balance
    )
    
    sharpe = calculate_sharpe(trades_df, min_date, max_date, starting_balance)
    sortino = calculate_sortino(trades_df, min_date, max_date, starting_balance)
    calmar = calculate_calmar(trades_df, min_date, max_date, starting_balance)
    expectancy, expectancy_ratio = calculate_expectancy(trades_df)
    
    gross_profit = trades_df[trades_df['profit_abs'] > 0]['profit_abs'].sum()
    gross_loss = abs(trades_df[trades_df['profit_abs'] < 0]['profit_abs'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    
    summary = pd.DataFrame({
        'Metric': [
            'Total Trades',
            'Winning Trades',
            'Losing Trades',
            'Win Rate %',
            'Total Profit',
            'Total Profit %', 
            'Avg Profit %',   
            'Avg Win %',
            'Avg Loss %',
            'Avg Duration (h)',
            'Sharpe Ratio',
            'Sortino Ratio',
            'Calmar Ratio',
            'Profit Factor',
            'Expectancy',
            'Expectancy Ratio',
            'Max Drawdown',
            'Max Drawdown %',
        ],
        'Value': [
            total_trades,
            winning_trades,
            losing_trades,
            f"{win_rate:.2f}%",
            f"{total_profit_abs:.2f}",
            f"{total_profit_pct:.2f}%",  
            f"{avg_profit:.2f}%",
            f"{avg_win:.2f}%",
            f"{avg_loss:.2f}%",
            f"{avg_duration:.1f}",
            f"{sharpe:.2f}",
            f"{sortino:.2f}",
            f"{calmar:.2f}",
            f"{profit_factor:.2f}",
            f"{expectancy:.2f}",
            f"{expectancy_ratio:.2f}",
            f"{drawdown.drawdown_abs:.2f}",
            f"{drawdown.relative_account_drawdown * 100:.2f}%",
        ]
    })
    
    if label:
        summary.columns = ['Metric', label]
    
    return summary


def plot_comparison_chart(backtest_df, live_df=None, metric='cumulative_profit', title=''):
    """Plot comparison chart between backtest and live results."""
    fig = go.Figure()
    
    backtest_sorted = backtest_df.sort_values('close_date').copy()
    backtest_sorted['cumulative_profit'] = backtest_sorted['profit_abs'].cumsum()
    
    fig.add_trace(go.Scatter(
        x=backtest_sorted['close_date'],
        y=backtest_sorted[metric],
        name='Backtest',
        line=dict(color='blue', width=2)
    ))
    
    if live_df is not None and len(live_df) > 0:
        live_sorted = live_df.sort_values('close_date').copy()
        live_sorted['cumulative_profit'] = live_sorted['profit_abs'].cumsum()
        
        fig.add_trace(go.Scatter(
            x=live_sorted['close_date'],
            y=live_sorted[metric],
            name='Live',
            line=dict(color='green', width=2)
        ))
    
    fig.update_layout(
        title=title or f'{metric.replace("_", " ").title()} - Backtest vs Live',
        height=500,
        xaxis_title='Date',
        yaxis_title=metric.replace('_', ' ').title()
    )
    
    return fig

## Performance Summary - Backtest

In [ ]:
backtest_summary = calculate_performance_summary(
    backtest_trades, 
    label="Backtest",
    starting_balance=config.get('dry_run_wallet', 1000)
)

display(backtest_summary)

In [ ]:
backtest_trades

## Performance Summary - Live (if available)

In [ ]:
def _fmt_roi_line(coin: float, mean_pct: float, sigma_pct: float) -> str:
    return f"{coin:.3f} {STAKE_CURRENCY} ({mean_pct:.2f}%) ({sigma_pct:.2f} Sigma%)"


if live_profit is not None:
    roi_closed = _fmt_roi_line(
        float(live_profit.get("profit_closed_coin", 0.0)),
        float(live_profit.get("profit_closed_percent_mean", 0.0)),
        float(live_profit.get("profit_closed_percent", 0.0)),
    )
    roi_all = _fmt_roi_line(
        float(live_profit.get("profit_all_coin", 0.0)),
        float(live_profit.get("profit_all_percent_mean", 0.0)),
        float(live_profit.get("profit_all_percent", 0.0)),
    )

    win = int(live_profit.get("winning_trades", 0))
    loss = int(live_profit.get("losing_trades", 0))

    live_summary = pd.DataFrame(
        {
            "Metric": [
                "ROI: Closed trades",
                "ROI: All trades",
                "Total Trade Count",
                "Win / Loss",
                "Winrate",
                "Profit factor",
                "Max Drawdown",
            ],
            "Live": [
                roi_closed,
                roi_all,
                int(live_profit.get("trade_count", 0)),
                f"{win} / {loss}",
                f"{float(live_profit.get('winrate', 0.0)) * 100:.2f}%",
                f"{float(live_profit.get('profit_factor', 0.0)):.2f}",
                f"{float(live_profit.get('max_drawdown_abs', 0.0)):.3f} {STAKE_CURRENCY} ({float(live_profit.get('max_drawdown', 0.0)) * 100:.2f}%)",
            ],
        }
    )

    display(live_summary)

elif live_trades is not None and len(live_trades) > 0:
    live_summary = calculate_performance_summary(
        live_trades,
        label="Live",
        starting_balance=config.get("dry_run_wallet", 100),
    )
    display(live_summary)

else:
    print("No live trades available for comparison")

## Side-by-Side Comparison

In [ ]:
# Side-by-side comparison MUST use the same metric names on both sides.

def _fmt_roi_line(coin: float, mean_pct: float, sigma_pct: float) -> str:
    return f"{coin:.3f} {STAKE_CURRENCY} ({mean_pct:.2f}%) ({sigma_pct:.2f} Sigma%)"


def _bt_roi_closed(trades_df: pd.DataFrame, starting_balance: float) -> str:
    coin = float(trades_df["profit_abs"].sum())
    mean_pct = float(trades_df["profit_ratio"].mean() * 100) if len(trades_df) else 0.0
    sigma_pct = float((coin / starting_balance) * 100) if starting_balance else 0.0
    return _fmt_roi_line(coin, mean_pct, sigma_pct)


if live_profit is not None:
    bt_starting_balance = float(config.get("dry_run_wallet", 1000))

    bt_roi_closed = _bt_roi_closed(backtest_trades, bt_starting_balance)
    bt_roi_all = bt_roi_closed

    live_roi_closed = _fmt_roi_line(
        float(live_profit.get("profit_closed_coin", 0.0)),
        float(live_profit.get("profit_closed_percent_mean", 0.0)),
        float(live_profit.get("profit_closed_percent", 0.0)),
    )
    live_roi_all = _fmt_roi_line(
        float(live_profit.get("profit_all_coin", 0.0)),
        float(live_profit.get("profit_all_percent_mean", 0.0)),
        float(live_profit.get("profit_all_percent", 0.0)),
    )

    bt_win = int((backtest_trades["profit_ratio"] > 0).sum())
    bt_loss = int((backtest_trades["profit_ratio"] <= 0).sum())

    live_win = int(live_profit.get("winning_trades", 0))
    live_loss = int(live_profit.get("losing_trades", 0))

    comparison = pd.DataFrame(
        {
            "Metric": [
                "ROI: Closed trades",
                "ROI: All trades",
                "Total Trade Count",
                "Closed Trade Count",
                "Win / Loss",
                "Winrate",
            ],
            "Backtest": [
                bt_roi_closed,
                bt_roi_all,
                int(len(backtest_trades)),
                int(len(backtest_trades)),
                f"{bt_win} / {bt_loss}",
                f"{(bt_win/len(backtest_trades)*100 if len(backtest_trades) else 0.0):.2f}%",
            ],
            "Live": [
                live_roi_closed,
                live_roi_all,
                int(live_profit.get("trade_count", 0)),
                int(live_profit.get("closed_trade_count", 0)),
                f"{live_win} / {live_loss}",
                f"{float(live_profit.get('winrate', 0.0)) * 100:.2f}%",
            ],
        }
    )

    display(comparison)

else:
    print("Live /profit not available - showing backtest only")
    display(backtest_summary)

## Cumulative Profit Comparison

In [ ]:
# Backtest-style metrics tables, aligned to Freqtrade /profit + /balance.

def _patch_metric(df: pd.DataFrame, metric: str, value) -> None:
    m = df["Metric"] == metric
    if m.any():
        df.loc[m, df.columns[1]] = value


def _build_live_summary_from_api(mode: str, *, live_closed: pd.DataFrame, starting_balance: float) -> pd.DataFrame:
    """Return a backtest-style summary, but with truth metrics patched from /api/v1/profit."""
    base = calculate_performance_summary(live_closed, label="Live", starting_balance=starting_balance)

    if mode == "closed":
        total_trades = int(live_profit.get("closed_trade_count", 0))
        total_profit = float(live_profit.get("profit_closed_coin", 0.0))
        total_profit_pct = float(live_profit.get("profit_closed_percent", 0.0))
        avg_profit_pct = float(live_profit.get("profit_closed_percent_mean", 0.0))
    elif mode == "all":
        total_trades = int(live_profit.get("trade_count", 0))
        total_profit = float(live_profit.get("profit_all_coin", 0.0))
        total_profit_pct = float(live_profit.get("profit_all_percent", 0.0))
        avg_profit_pct = float(live_profit.get("profit_all_percent_mean", 0.0))
    else:
        raise ValueError("mode must be 'closed' or 'all'")

    _patch_metric(base, "Total Trades", total_trades)
    _patch_metric(base, "Winning Trades", int(live_profit.get("winning_trades", 0)))
    _patch_metric(base, "Losing Trades", int(live_profit.get("losing_trades", 0)))
    _patch_metric(base, "Win Rate %", f"{float(live_profit.get('winrate', 0.0)) * 100:.2f}%")
    _patch_metric(base, "Total Profit", f"{total_profit:.2f}")
    _patch_metric(base, "Total Profit %", f"{total_profit_pct:.2f}%")
    _patch_metric(base, "Avg Profit %", f"{avg_profit_pct:.2f}%")
    _patch_metric(base, "Profit Factor", f"{float(live_profit.get('profit_factor', 0.0)):.2f}")
    _patch_metric(base, "Expectancy", f"{float(live_profit.get('expectancy', 0.0)):.2f}")
    _patch_metric(base, "Expectancy Ratio", f"{float(live_profit.get('expectancy_ratio', 0.0)):.2f}")
    _patch_metric(base, "Max Drawdown", f"{float(live_profit.get('max_drawdown_abs', 0.0)):.2f}")
    _patch_metric(base, "Max Drawdown %", f"{float(live_profit.get('max_drawdown', 0.0)) * 100:.2f}%")

    return base


if live_profit is not None and live_trades is not None and len(live_trades) > 0:
    live_closed = live_trades[live_trades.get("is_open", False) == False].copy()
    if len(live_closed) == 0:
        live_closed = live_trades[live_trades["close_date"].notna()].copy()

    bt_starting_balance = float(config.get("dry_run_wallet", 1000))
    live_starting_balance = bt_starting_balance
    if isinstance(live_balance, dict) and live_balance.get("starting_capital") is not None:
        live_starting_balance = float(live_balance.get("starting_capital"))

    bt_closed = calculate_performance_summary(backtest_trades, label="Backtest", starting_balance=bt_starting_balance)
    bt_all = bt_closed.copy()

    live_closed_sum = _build_live_summary_from_api(
        "closed",
        live_closed=live_closed,
        starting_balance=live_starting_balance,
    )
    live_all_sum = _build_live_summary_from_api(
        "all",
        live_closed=live_closed,
        starting_balance=live_starting_balance,
    )

    print("\n### Backtest vs Live -- CLOSED trades")
    table_closed = bt_closed.merge(live_closed_sum, on="Metric", how="left")
    display(table_closed)

    print("\n### Backtest vs Live -- ALL trades (ROI rows are mark-to-market)")
    table_all = bt_all.merge(live_all_sum, on="Metric", how="left")
    display(table_all)

else:
    print("Live /profit + trades not available - cannot build aligned backtest-style tables.")

## 6. Performance Metrics Summary

Comprehensive side-by-side comparison of all key metrics.

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    bt_sorted = backtest_trades.sort_values('close_date').copy()
    bt_sorted['cumulative_profit'] = bt_sorted['profit_abs'].cumsum()
    bt_sorted['running_max'] = bt_sorted['cumulative_profit'].cummax()
    bt_sorted['drawdown'] = bt_sorted['cumulative_profit'] - bt_sorted['running_max']
    
    live_sorted = live_trades.sort_values('close_date').copy()
    live_sorted['cumulative_profit'] = live_sorted['profit_abs'].cumsum()
    live_sorted['running_max'] = live_sorted['cumulative_profit'].cummax()
    live_sorted['drawdown'] = live_sorted['cumulative_profit'] - live_sorted['running_max']
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=bt_sorted['close_date'],
        y=bt_sorted['drawdown'],
        fill='tozeroy',
        name='Backtest DD',
        line=dict(color='blue'),
        fillcolor='rgba(0,0,255,0.3)'
    ))
    
    fig.add_trace(go.Scatter(
        x=live_sorted['close_date'],
        y=live_sorted['drawdown'],
        fill='tozeroy',
        name='Live DD',
        line=dict(color='green'),
        fillcolor='rgba(0,255,0,0.3)'
    ))
    
    fig.update_layout(
        title='Drawdown Comparison: Backtest vs Live',
        xaxis_title='Date',
        yaxis_title='Drawdown',
        height=500,
        hovermode='x unified'
    )
    
    fig.show()
    
    bt_max_dd = bt_sorted['drawdown'].min()
    live_max_dd = live_sorted['drawdown'].min()
    
    print(f"\nDrawdown Comparison:")
    print(f"\nBacktest Max Drawdown: {bt_max_dd:.2f} {STAKE_CURRENCY}")
    print(f"Live Max Drawdown: {live_max_dd:.2f} {STAKE_CURRENCY}")
    print(f"Difference: {(live_max_dd - bt_max_dd):.2f} {STAKE_CURRENCY}")
    
    if abs(live_max_dd) < abs(bt_max_dd):
        print(f"\nLive trading has BETTER drawdown control ({abs(bt_max_dd - live_max_dd):.2f} {STAKE_CURRENCY} less)")
    elif abs(live_max_dd) > abs(bt_max_dd):
        print(f"\nLive trading has WORSE drawdown ({abs(live_max_dd - bt_max_dd):.2f} {STAKE_CURRENCY} more)")
    else:
        print("\nDrawdowns are similar")
else:
    print("No live trades for comparison")

## 5. Cumulative Profit & Drawdown Comparison

Compare drawdown patterns between backtest and live trading.

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    bt_starting_balance = float(config.get('dry_run_wallet', 1000))

    live_starting_balance = None

    if isinstance(live_balance, dict) and live_balance.get('starting_capital') is not None:
        live_starting_balance = float(live_balance.get('starting_capital'))

    if live_starting_balance is None and isinstance(live_balance, dict) and live_balance.get('total') is not None:
        current_total = float(live_balance['total'])
        closed_profit = live_trades['profit_abs'].sum()
        live_starting_balance = current_total - closed_profit

    if live_starting_balance is None and isinstance(live_profit, dict) and live_profit.get('starting_balance') is not None:
        live_starting_balance = float(live_profit['starting_balance'])

    if live_starting_balance is None:
        live_starting_balance = bt_starting_balance
        print("Warning: Could not determine live starting balance, using backtest value")

    print(f"Starting balances: Backtest={bt_starting_balance:.2f} {STAKE_CURRENCY}, Live={live_starting_balance:.2f} {STAKE_CURRENCY}")

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            'Cumulative Profit Comparison (% of starting balance)',
            'Daily Profit (Backtest)',
            'Daily Profit (Live)'
        ),
        vertical_spacing=0.12,
        row_heights=[0.5, 0.25, 0.25]
    )

    bt_sorted = backtest_trades.sort_values('close_date').copy()
    bt_sorted = bt_sorted.reset_index(drop=True)
    bt_sorted['portfolio_return_pct'] = 0.0
    bt_current_balance = bt_starting_balance
    
    for idx in range(len(bt_sorted)):
        if bt_current_balance > 0 and bt_sorted.loc[idx, 'stake_amount'] is not None:
            stake_pct = bt_sorted.loc[idx, 'stake_amount'] / bt_current_balance
            portfolio_return = bt_sorted.loc[idx, 'profit_ratio'] * stake_pct * 100.0
            bt_sorted.loc[idx, 'portfolio_return_pct'] = portfolio_return
        bt_current_balance = bt_current_balance + bt_sorted.loc[idx, 'profit_abs']
    
    bt_sorted['cumulative_profit_pct'] = bt_sorted['portfolio_return_pct'].cumsum()
    last_bt_date = bt_sorted['close_date'].max()

    live_sorted = live_trades.sort_values('close_date').copy()
    live_sorted = live_sorted.reset_index(drop=True)
    live_sorted['portfolio_return_pct'] = 0.0
    live_current_balance = live_starting_balance
    
    for idx in range(len(live_sorted)):
        if live_current_balance > 0:
            stake_amount = live_sorted.loc[idx, 'stake_amount']
            
            if stake_amount is None or pd.isna(stake_amount) or stake_amount <= 0:
                profit_ratio = live_sorted.loc[idx, 'profit_ratio']
                profit_abs = live_sorted.loc[idx, 'profit_abs']
                if profit_ratio != 0 and not pd.isna(profit_ratio):
                    stake_amount = abs(profit_abs / profit_ratio)
                else:
                    stake_amount = None
            
            if stake_amount is not None and stake_amount > 0:
                stake_pct = stake_amount / live_current_balance
                portfolio_return = live_sorted.loc[idx, 'profit_ratio'] * stake_pct * 100.0
                live_sorted.loc[idx, 'portfolio_return_pct'] = portfolio_return
            else:
                portfolio_return = (live_sorted.loc[idx, 'profit_abs'] / live_current_balance) * 100.0
                live_sorted.loc[idx, 'portfolio_return_pct'] = portfolio_return
        
        live_current_balance = live_current_balance + live_sorted.loc[idx, 'profit_abs']
    
    live_sorted['cumulative_profit_pct'] = live_sorted['portfolio_return_pct'].cumsum()

    fig.add_trace(
        go.Scatter(
            x=bt_sorted['close_date'],
            y=bt_sorted['cumulative_profit_pct'],
            name='Backtest',
            line=dict(color='blue', width=2),
            mode='lines+markers'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=live_sorted['close_date'],
            y=live_sorted['cumulative_profit_pct'],
            name='Live',
            line=dict(color='green', width=2),
            mode='lines+markers'
        ),
        row=1, col=1
    )

    bt_sorted['close_date_normalized'] = pd.to_datetime(bt_sorted['close_date']).dt.tz_localize(None).dt.normalize()
    live_sorted['close_date_normalized'] = pd.to_datetime(live_sorted['close_date']).dt.tz_localize(None).dt.normalize()

    bt_daily = bt_sorted.groupby('close_date_normalized')['profit_ratio'].sum().reset_index()
    bt_daily.columns = ['date', 'profit_pct']
    bt_daily['profit_pct'] = bt_daily['profit_pct'] * 100.0
    bt_daily['date'] = pd.to_datetime(bt_daily['date']).dt.normalize()

    live_daily = live_sorted.groupby('close_date_normalized')['profit_ratio'].sum().reset_index()
    live_daily.columns = ['date', 'profit_pct']
    live_daily['profit_pct'] = live_daily['profit_pct'] * 100.0
    live_daily['date'] = pd.to_datetime(live_daily['date']).dt.normalize()

    bt_trade_dates = pd.to_datetime(bt_sorted['close_date']).dt.tz_localize(None).dt.normalize().tolist()
    live_trade_dates = pd.to_datetime(live_sorted['close_date']).dt.tz_localize(None).dt.normalize().tolist()
    all_trade_dates = sorted(set(bt_trade_dates + live_trade_dates))
    all_daily_dates = sorted(set(bt_daily['date'].tolist() + live_daily['date'].tolist()))
    all_dates = sorted(set(all_trade_dates + all_daily_dates))
    min_date = min(all_dates) if all_dates else None
    max_date = max(all_dates) if all_dates else None
    
    if min_date and max_date:
        min_date = min_date - pd.Timedelta(days=2)
        max_date = max_date + pd.Timedelta(days=2)
    
    if min_date and max_date:
        date_range = pd.date_range(start=min_date, end=max_date, freq='D')
        bt_daily_aligned = pd.DataFrame({'date': date_range})
        bt_daily_aligned = bt_daily_aligned.merge(bt_daily, on='date', how='left').fillna(0)
        bt_daily_aligned = bt_daily_aligned.sort_values('date')
        
        live_daily_aligned = pd.DataFrame({'date': date_range})
        live_daily_aligned = live_daily_aligned.merge(live_daily, on='date', how='left').fillna(0)
        live_daily_aligned = live_daily_aligned.sort_values('date')
    else:
        bt_daily_aligned = bt_daily
        live_daily_aligned = live_daily

    fig.add_trace(
        go.Bar(
            x=bt_daily_aligned['date'],
            y=bt_daily_aligned['profit_pct'],
            name='Backtest Daily Profit',
            marker_color='blue',
            showlegend=False,
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Bar(
            x=live_daily_aligned['date'],
            y=live_daily_aligned['profit_pct'],
            name='Live Daily Profit',
            marker_color='green',
            showlegend=False,
        ),
        row=3, col=1
    )

    fig.update_layout(
        height=1000,
        title='Cumulative Profit Comparison',
        showlegend=True,
        hovermode='x unified'
    )

    if min_date and max_date:
        fig.update_xaxes(title_text='Date', range=[min_date, max_date], row=1, col=1)
        fig.update_xaxes(title_text='Date', range=[min_date, max_date], row=2, col=1)
        fig.update_xaxes(title_text='Date', range=[min_date, max_date], row=3, col=1)
    else:
        fig.update_xaxes(title_text='Date', row=1, col=1)
        fig.update_xaxes(title_text='Date', row=2, col=1)
        fig.update_xaxes(title_text='Date', row=3, col=1)
    
    fig.update_yaxes(title_text='Cumulative Profit (%)', row=1, col=1)
    fig.update_yaxes(title_text='Daily Profit (%)', row=2, col=1)
    fig.update_yaxes(title_text='Daily Profit (%)', row=3, col=1)

    fig.add_annotation(
        x=last_bt_date,
        y=bt_sorted['cumulative_profit_pct'].iloc[-1],
        text='Last Backtest Trade',
        showarrow=True,
        arrowhead=2,
        arrowcolor='blue',
        ax=40,
        ay=-40,
        row=1,
        col=1,
    )

    fig.show()

    print(f"\nEquity Curve Summary (%, normalized):")
    print(f"Backtest start balance: {bt_starting_balance:.2f} {STAKE_CURRENCY}")
    print(f"Live start balance: {live_starting_balance:.2f} {STAKE_CURRENCY}")
    print(f"Backtest final: {bt_sorted['cumulative_profit_pct'].iloc[-1]:+.2f}%")
    print(f"Live final: {live_sorted['cumulative_profit_pct'].iloc[-1]:+.2f}%")

else:
    print("No live trades for comparison")

## 4. Equity Curve Comparison

Side-by-side comparison of cumulative profit curves.

In [ ]:
# Calculate common pairs first
if live_trades is not None and len(live_trades) > 0:
    common_pairs = list(set(backtest_trades['pair'].unique()) & set(live_trades['pair'].unique()))
else:
    common_pairs = []

if live_trades is not None and len(live_trades) > 0 and len(common_pairs) > 0:
    print("Entry/Exit Price Analysis for Common Pairs:\n")
    
    price_comparison_data = []
    
    for pair in common_pairs:
        bt_pair = backtest_trades[backtest_trades['pair'] == pair]
        live_pair = live_trades[live_trades['pair'] == pair]
        
        bt_avg_open = bt_pair['open_rate'].mean()
        live_avg_open = live_pair['open_rate'].mean()
        bt_avg_close = bt_pair['close_rate'].mean()
        live_avg_close = live_pair['close_rate'].mean()
        
        open_diff_pct = ((live_avg_open - bt_avg_open) / bt_avg_open * 100) if bt_avg_open != 0 else 0
        close_diff_pct = ((live_avg_close - bt_avg_close) / bt_avg_close * 100) if bt_avg_close != 0 else 0
        
        price_comparison_data.append({
            'Pair': pair,
            'BT Avg Open': f"{bt_avg_open:.6f}",
            'Live Avg Open': f"{live_avg_open:.6f}",
            'Open Diff %': f"{open_diff_pct:.2f}%",
            'BT Avg Close': f"{bt_avg_close:.6f}",
            'Live Avg Close': f"{live_avg_close:.6f}",
            'Close Diff %': f"{close_diff_pct:.2f}%",
            'BT Avg Profit %': f"{bt_pair['profit_ratio'].mean() * 100:.2f}%",
            'Live Avg Profit %': f"{live_pair['profit_ratio'].mean() * 100:.2f}%",
        })
    
    price_comparison_df = pd.DataFrame(price_comparison_data)
    display(price_comparison_df)
    
    print("\nKey Insights:")
    print("  Open Diff %: Difference in average entry price (positive = live entered higher)")
    print("  Close Diff %: Difference in average exit price (positive = live exited higher)")
    print("  Large differences may indicate slippage, timing differences, or execution delays")

elif live_trades is not None and len(live_trades) > 0:
    print("No common pairs to compare entry/exit prices")
else:
    print("No live trades for comparison")

## 3. Entry/Exit Price Comparison

Analyze differences in open_rate and close_rate between backtest and live for common pairs.

In [ ]:
# ==============================================================================
# Trade Matching & Delay Analysis Functions
# ==============================================================================

import pandas as pd
import numpy as np
from datetime import timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def match_trades(backtest_df, live_df, time_window_hours=4):
    """
    Match trades between backtest and live based on:
    - Same pair
    - Same direction (is_short)
    - Open time within specified window
    """
    bt = backtest_df.copy()
    live = live_df.copy()
    
    for col in ['open_date', 'close_date']:
        if bt[col].dt.tz is not None:
            bt[col] = bt[col].dt.tz_localize(None)
        if live[col].dt.tz is not None:
            live[col] = live[col].dt.tz_localize(None)
    
    bt = bt.add_prefix('bt_')
    live = live.add_prefix('live_')
    
    matched_trades = []
    used_live_indices = set()
    
    for bt_idx, bt_row in bt.iterrows():
        candidates = live[
            (live['live_pair'] == bt_row['bt_pair']) &
            (live['live_is_short'] == bt_row['bt_is_short']) &
            (~live.index.isin(used_live_indices))
        ].copy()
        
        if len(candidates) == 0:
            continue
        
        candidates['time_diff'] = abs(
            (candidates['live_open_date'] - bt_row['bt_open_date']).dt.total_seconds() / 3600
        )
        candidates = candidates[candidates['time_diff'] <= time_window_hours]
        
        if len(candidates) == 0:
            continue
        
        best_match_idx = candidates['time_diff'].idxmin()
        live_row = candidates.loc[best_match_idx]
        used_live_indices.add(best_match_idx)
        
        matched = {**bt_row.to_dict(), **live_row.to_dict()}
        matched['bt_idx'] = bt_idx
        matched['live_idx'] = best_match_idx
        matched_trades.append(matched)
    
    if len(matched_trades) == 0:
        print("WARNING: No matching trades found within the time window")
        return pd.DataFrame()
    
    matched_df = pd.DataFrame(matched_trades)
    
    matched_df['entry_delay_seconds'] = (
        matched_df['live_open_date'] - matched_df['bt_open_date']
    ).dt.total_seconds()
    matched_df['entry_delay_minutes'] = matched_df['entry_delay_seconds'] / 60
    matched_df['entry_delay_hours'] = matched_df['entry_delay_minutes'] / 60
    
    has_both_closed = matched_df['live_close_date'].notna() & matched_df['bt_close_date'].notna()
    matched_df['exit_delay_seconds'] = np.nan
    matched_df.loc[has_both_closed, 'exit_delay_seconds'] = (
        matched_df.loc[has_both_closed, 'live_close_date'] - 
        matched_df.loc[has_both_closed, 'bt_close_date']
    ).dt.total_seconds()
    matched_df['exit_delay_minutes'] = matched_df['exit_delay_seconds'] / 60
    matched_df['exit_delay_hours'] = matched_df['exit_delay_minutes'] / 60
    
    return matched_df


def format_timedelta(seconds):
    """Format seconds as human-readable string."""
    if pd.isna(seconds):
        return "N/A"
    sign = "+" if seconds >= 0 else "-"
    seconds = abs(seconds)
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    
    if hours > 0:
        return f"{sign}{hours}h {minutes}m {secs}s"
    elif minutes > 0:
        return f"{sign}{minutes}m {secs}s"
    else:
        return f"{sign}{secs}s"


def get_outlier_bounds(data, method='iqr', multiplier=1.5):
    """Get outlier bounds for a data series."""
    if method == 'iqr':
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        return Q1 - multiplier * IQR, Q3 + multiplier * IQR
    elif method == 'percentile':
        return data.quantile(0.05), data.quantile(0.95)
    elif method == 'std':
        return data.mean() - multiplier * data.std(), data.mean() + multiplier * data.std()


def filter_outliers(df, column, method='iqr', multiplier=1.5):
    """Filter outliers from DataFrame based on a column."""
    data = df[column].dropna()
    lower, upper = get_outlier_bounds(data, method, multiplier)
    mask = (df[column] >= lower) & (df[column] <= upper)
    mask = mask | df[column].isna()
    return df[mask], lower, upper


def display_delay_stats(matched_df, method='iqr', multiplier=1.5):
    """Display entry and exit delay statistics with and without outliers."""
    print("=" * 90)
    print("TRADE TIMING DELAY STATISTICS: ALL vs WITHOUT OUTLIERS")
    print("=" * 90)
    print(f"\nTotal matched trades: {len(matched_df)}")
    print(f"Outlier method: {method.upper()} (multiplier={multiplier})")
    print()
    
    def calc_stats(data, label):
        if len(data) == 0:
            return None
        return {
            'Category': label,
            'Count': len(data),
            'Mean (min)': round(data.mean() / 60, 2),
            'Median (min)': round(data.median() / 60, 2),
            'Std Dev (min)': round(data.std() / 60, 2),
            'Min (min)': round(data.min() / 60, 2),
            'Max (min)': round(data.max() / 60, 2),
            'Mean': format_timedelta(data.mean()),
            'Median': format_timedelta(data.median()),
        }
    
    stats_rows = []
    
    entry_all = matched_df['entry_delay_seconds'].dropna()
    if len(entry_all) > 0:
        stats_rows.append(calc_stats(entry_all, 'ENTRY - All'))
        
        lower, upper = get_outlier_bounds(entry_all, method, multiplier)
        entry_filtered = entry_all[(entry_all >= lower) & (entry_all <= upper)]
        entry_stats = calc_stats(entry_filtered, 'ENTRY - No Outliers')
        if entry_stats:
            entry_stats['Outliers Removed'] = len(entry_all) - len(entry_filtered)
            stats_rows.append(entry_stats)
    
    stats_rows.append({k: '---' for k in ['Category', 'Count', 'Outliers Removed', 'Mean (min)', 
                                           'Median (min)', 'Std Dev (min)', 'Min (min)', 'Max (min)', 
                                           'Mean', 'Median']})
    
    exit_all = matched_df['exit_delay_seconds'].dropna()
    if len(exit_all) > 0:
        stats_rows.append(calc_stats(exit_all, 'EXIT - All'))
        
        lower, upper = get_outlier_bounds(exit_all, method, multiplier)
        exit_filtered = exit_all[(exit_all >= lower) & (exit_all <= upper)]
        exit_stats = calc_stats(exit_filtered, 'EXIT - No Outliers')
        if exit_stats:
            exit_stats['Outliers Removed'] = len(exit_all) - len(exit_filtered)
            stats_rows.append(exit_stats)
    
    stats_df = pd.DataFrame(stats_rows)
    col_order = ['Category', 'Count', 'Outliers Removed', 'Mean', 'Median', 
                 'Mean (min)', 'Median (min)', 'Std Dev (min)', 'Min (min)', 'Max (min)']
    stats_df = stats_df[[c for c in col_order if c in stats_df.columns]]
    display(stats_df.set_index('Category'))
    
    print("\n" + "-" * 50)
    print("TIMING DIRECTION BREAKDOWN")
    print("-" * 50)
    
    if len(entry_all) > 0:
        late = (entry_all > 0).sum()
        early = (entry_all < 0).sum()
        print(f"\nEntry timing ({len(entry_all)} trades):")
        print(f"  Live LATER than backtest:   {late:3d} ({late/len(entry_all)*100:5.1f}%)")
        print(f"  Live EARLIER than backtest: {early:3d} ({early/len(entry_all)*100:5.1f}%)")
    
    if len(exit_all) > 0:
        late = (exit_all > 0).sum()
        early = (exit_all < 0).sum()
        print(f"\nExit timing ({len(exit_all)} closed trades):")
        print(f"  Live LATER than backtest:   {late:3d} ({late/len(exit_all)*100:5.1f}%)")
        print(f"  Live EARLIER than backtest: {early:3d} ({early/len(exit_all)*100:5.1f}%)")


def create_side_by_side_table(matched_df):
    """Create a side-by-side comparison table of matched trades."""
    return pd.DataFrame({
        'Pair': matched_df['bt_pair'],
        'Direction': matched_df['bt_is_short'].map({True: 'SHORT', False: 'LONG'}),
        'BT Entry': matched_df['bt_open_date'].dt.strftime('%Y-%m-%d %H:%M'),
        'Live Entry': matched_df['live_open_date'].dt.strftime('%Y-%m-%d %H:%M:%S'),
        'Entry Delta': matched_df['entry_delay_seconds'].apply(format_timedelta),
        'Entry Delta (min)': matched_df['entry_delay_minutes'].round(2),
        'BT Exit': matched_df['bt_close_date'].dt.strftime('%Y-%m-%d %H:%M'),
        'Live Exit': matched_df['live_close_date'].apply(
            lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else 'OPEN'
        ),
        'Exit Delta': matched_df['exit_delay_seconds'].apply(format_timedelta),
        'Exit Delta (min)': matched_df['exit_delay_minutes'].round(2),
        'BT Profit %': matched_df['bt_profit_ratio_pct'].round(2),
        'Live Profit %': matched_df['live_profit_ratio_pct'].round(2),
    })


def identify_outliers_table(matched_df, column='exit_delay_minutes', method='iqr', multiplier=1.5):
    """Show trades that are outliers."""
    data = matched_df[matched_df[column].notna()].copy()
    lower, upper = get_outlier_bounds(data[column], method, multiplier)
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    
    if len(outliers) == 0:
        print(f"No outliers found for {column}")
        return None
    
    return pd.DataFrame({
        'Pair': outliers['bt_pair'],
        'Direction': outliers['bt_is_short'].map({True: 'SHORT', False: 'LONG'}),
        'BT Exit': outliers['bt_close_date'].dt.strftime('%Y-%m-%d %H:%M'),
        'Live Exit': outliers['live_close_date'].dt.strftime('%Y-%m-%d %H:%M'),
        'Exit Delay (min)': outliers['exit_delay_minutes'].round(2),
        'Exit Delay (hours)': outliers['exit_delay_hours'].round(2),
        'Exit Delay': outliers['exit_delay_seconds'].apply(format_timedelta),
    }).sort_values('Exit Delay (min)', key=abs, ascending=False)


def plot_delay_distribution(matched_df, remove_outliers=True, method='iqr', multiplier=1.5):
    """Create visualization of entry and exit delays."""
    entry_df = matched_df.copy()
    exit_df = matched_df[matched_df['exit_delay_minutes'].notna()].copy()
    
    if remove_outliers:
        entry_df, el, eu = filter_outliers(entry_df, 'entry_delay_minutes', method, multiplier)
        exit_df, xl, xu = filter_outliers(exit_df, 'exit_delay_minutes', method, multiplier)
        print(f"Outlier filtering: Entry kept [{el:.1f}, {eu:.1f}] min | Exit kept [{xl:.1f}, {xu:.1f}] min")
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f'Entry Delay Distribution (min){" [filtered]" if remove_outliers else ""}',
            f'Exit Delay Distribution (min){" [filtered]" if remove_outliers else ""}',
            'Entry Delay Over Time',
            'Exit Delay Over Time'
        ),
        vertical_spacing=0.15, horizontal_spacing=0.1
    )
    
    fig.add_trace(go.Histogram(x=entry_df['entry_delay_minutes'], nbinsx=30, 
                                marker_color='blue', opacity=0.7), row=1, col=1)
    fig.add_trace(go.Histogram(x=exit_df['exit_delay_minutes'], nbinsx=30, 
                                marker_color='green', opacity=0.7), row=1, col=2)
    
    fig.add_trace(go.Scatter(
        x=entry_df['bt_open_date'], y=entry_df['entry_delay_minutes'],
        mode='markers', marker=dict(color='blue', size=8),
        hovertemplate="<b>%{customdata}</b><br>%{x}<br>%{y:.1f} min<extra></extra>",
        customdata=entry_df['bt_pair']
    ), row=2, col=1)
    
    fig.add_trace(go.Scatter(
        x=exit_df['bt_close_date'], y=exit_df['exit_delay_minutes'],
        mode='markers', marker=dict(color='green', size=8),
        hovertemplate="<b>%{customdata}</b><br>%{x}<br>%{y:.1f} min<extra></extra>",
        customdata=exit_df['bt_pair']
    ), row=2, col=2)
    
    for col in [1, 2]:
        fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5, row=2, col=col)
    
    title = 'Trade Timing Delay: Backtest vs Live'
    if remove_outliers:
        title += f' (Outliers Removed)'
    
    fig.update_layout(title=title, height=700, showlegend=False)
    fig.update_xaxes(title_text="Delay (min)", row=1, col=1)
    fig.update_xaxes(title_text="Delay (min)", row=1, col=2)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_yaxes(title_text="Delay (min)", row=2, col=1)
    fig.update_yaxes(title_text="Delay (min)", row=2, col=2)
    
    return fig

In [ ]:
# ==============================================================================
# Run Trade Delay Analysis
# ==============================================================================

OUTLIER_METHOD = 'iqr'      # 'iqr', 'percentile', or 'std'
OUTLIER_MULTIPLIER = 1.5    # 1.5 = standard, 3.0 = less aggressive
TIME_WINDOW_HOURS = 4       # Max hours difference to match trades

if live_trades is not None and len(live_trades) > 0:
    matched_trades_df = match_trades(backtest_trades, live_trades, time_window_hours=TIME_WINDOW_HOURS)
    
    if len(matched_trades_df) > 0:
        display_delay_stats(matched_trades_df, method=OUTLIER_METHOD, multiplier=OUTLIER_MULTIPLIER)
        
        print("\n" + "=" * 90)
        print("SIDE-BY-SIDE TRADE COMPARISON")
        print("=" * 90)
        comparison_table = create_side_by_side_table(matched_trades_df)
        display(comparison_table.sort_values('Entry Delta (min)', ascending=False))
        
        print("\n" + "=" * 90)
        print("EXIT DELAY OUTLIERS")
        print("=" * 90)
        outliers_table = identify_outliers_table(matched_trades_df, 'exit_delay_minutes', 
                                                  method=OUTLIER_METHOD, multiplier=OUTLIER_MULTIPLIER)
        if outliers_table is not None:
            display(outliers_table)
        
        print("\n" + "=" * 90)
        print("DELAY DISTRIBUTIONS (OUTLIERS REMOVED)")
        print("=" * 90)
        fig = plot_delay_distribution(matched_trades_df, remove_outliers=True, 
                                       method=OUTLIER_METHOD, multiplier=OUTLIER_MULTIPLIER)
        fig.show()
        
        print("\n" + "=" * 90)
        print("DELAY DISTRIBUTIONS (ALL DATA)")
        print("=" * 90)
        fig_all = plot_delay_distribution(matched_trades_df, remove_outliers=False)
        fig_all.show()
else:
    print("No live trades available for comparison")

In [ ]:
if live_trades is not None and len(live_trades) > 0 and 'outliers_table' in dir() and outliers_table is not None:
    outliers_table.sort_values(by='Live Exit')

In [ ]:
show_latest_trades = True

import pandas as pd

cutoff_utc = pd.Timestamp.now(tz="UTC").normalize() - pd.Timedelta(days=7)
cutoff_naive = cutoff_utc.tz_localize(None)
if show_latest_trades:
    backtest_trades = backtest_trades[backtest_trades.close_date > cutoff_utc]
    if live_trades is not None:
        live_trades = live_trades[live_trades.close_date > cutoff_naive]

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    backtest_pairs = set(backtest_trades['pair'].unique())
    live_pairs = set(live_trades['pair'].unique())
    common_pairs = backtest_pairs & live_pairs
    all_pairs = backtest_pairs | live_pairs
    
    print(f"Pair Overlap Analysis:")
    print(f"  Backtest pairs: {len(backtest_pairs)}")
    print(f"  Live pairs: {len(live_pairs)}")
    print(f"  Common pairs: {len(common_pairs)}")
    print(f"  Total unique pairs: {len(all_pairs)}")
    
    comparison_data = []
    
    for pair in all_pairs:
        bt_pair = backtest_trades[backtest_trades['pair'] == pair]
        live_pair = live_trades[live_trades['pair'] == pair]
        
        bt_trades_count = len(bt_pair) if len(bt_pair) > 0 else 0
        bt_avg_profit = bt_pair['profit_ratio'].mean() * 100 if len(bt_pair) > 0 else None
        bt_win_rate = (len(bt_pair[bt_pair['profit_ratio'] > 0]) / len(bt_pair) * 100) if len(bt_pair) > 0 else None
        bt_avg_duration = bt_pair['duration_hours'].mean() if len(bt_pair) > 0 else None
        
        live_trades_count = len(live_pair) if len(live_pair) > 0 else 0
        live_avg_profit = live_pair['profit_ratio'].mean() * 100 if len(live_pair) > 0 else None
        live_win_rate = (len(live_pair[live_pair['profit_ratio'] > 0]) / len(live_pair) * 100) if len(live_pair) > 0 else None
        live_avg_duration = live_pair['duration_hours'].mean() if len(live_pair) > 0 else None
        
        profit_diff = (live_avg_profit - bt_avg_profit) if (live_avg_profit is not None and bt_avg_profit is not None) else None
        duration_diff = (live_avg_duration - bt_avg_duration) if (live_avg_duration is not None and bt_avg_duration is not None) else None
        
        if pair in common_pairs:
            status = 'Both'
        elif pair in backtest_pairs:
            status = 'Missed in Live'
        else:
            status = 'Live Only'
        
        comparison_data.append({
            'Pair': pair,
            'Status': status,
            'BT Trades': bt_trades_count,
            'Live Trades': live_trades_count,
            'BT Avg Profit %': bt_avg_profit,
            'Live Avg Profit %': live_avg_profit,
            'Profit Diff %': profit_diff,
            'BT Win Rate %': bt_win_rate,
            'Live Win Rate %': live_win_rate,
            'BT Avg Duration (h)': bt_avg_duration,
            'Live Avg Duration (h)': live_avg_duration,
            'Duration Diff (h)': duration_diff,
        })
    
    comparison_df = pd.DataFrame(comparison_data).round(2)
    comparison_df = comparison_df.sort_values('Status', ascending=True)
    
    print("\nComplete Pair-by-Pair Performance Comparison:")
    display(comparison_df)
    
    fig = go.Figure()
    
    pairs_list = comparison_df['Pair'].tolist()
    bt_profits = [p if p is not None else 0 for p in comparison_df['BT Avg Profit %']]
    live_profits = [p if p is not None else 0 for p in comparison_df['Live Avg Profit %']]
    
    bt_colors = ['blue' if comparison_df.iloc[i]['BT Trades'] > 0 else 'lightgray' for i in range(len(comparison_df))]
    live_colors = ['green' if comparison_df.iloc[i]['Live Trades'] > 0 else 'lightgray' for i in range(len(comparison_df))]
    
    fig.add_trace(go.Bar(
        x=pairs_list,
        y=bt_profits,
        name='Backtest',
        marker_color=bt_colors,
        text=[f"{p:.2f}%" if p != 0 else "N/A" for p in bt_profits],
        textposition='outside'
    ))
    
    fig.add_trace(go.Bar(
        x=pairs_list,
        y=live_profits,
        name='Live',
        marker_color=live_colors,
        text=[f"{p:.2f}%" if p != 0 else "N/A" for p in live_profits],
        textposition='outside'
    ))
    
    fig.update_layout(
        title='Average Profit % by Pair: Backtest vs Live (All Pairs)',
        xaxis_title='Pair',
        yaxis_title='Avg Profit %',
        barmode='group',
        height=500,
        xaxis={'categoryorder': 'total descending'}
    )
    
    fig.show()
    
    missed_in_live = comparison_df[comparison_df['Status'] == 'Missed in Live']
    live_only = comparison_df[comparison_df['Status'] == 'Live Only']
    
    if len(missed_in_live) > 0:
        print(f"\nMissed Opportunities in Live Trading ({len(missed_in_live)} pairs):")
        print(f"   {', '.join(sorted(missed_in_live['Pair']))}")
        missed_profit = missed_in_live['BT Avg Profit %'].mean()
        print(f"   Average backtest profit for missed pairs: {missed_profit:.2f}%")
    
    if len(live_only) > 0:
        print(f"\nPairs Traded Live but NOT in Backtest ({len(live_only)} pairs):")
        print(f"   {', '.join(sorted(live_only['Pair']))}")
        live_only_profit = live_only['Live Avg Profit %'].mean()
        print(f"   Average live profit for these pairs: {live_only_profit:.2f}%")
        
else:
    print("No live trades for comparison")

## Dynamic Pairlist Analysis (Vault Mode)

For vault mode, shows which pairs the backtest's HistoricalVolumePairList selected
vs what the live bot actually traded. Helps identify pairlist divergence.

In [ ]:
# Dynamic pairlist analysis - only meaningful for vault mode
if BOT_MODE == "vault" and live_trades is not None and len(live_trades) > 0:
    print("=" * 70)
    print("DYNAMIC PAIRLIST ANALYSIS")
    print("=" * 70)
    
    # Get daily traded pairs from backtest
    bt_daily_pairs = (
        backtest_trades
        .assign(open_day=pd.to_datetime(backtest_trades["open_date"]).dt.date)
        .groupby("open_day")["pair"]
        .apply(set)
        .to_dict()
    )
    
    # Get daily traded pairs from live
    live_daily_pairs = (
        live_trades
        .assign(open_day=pd.to_datetime(live_trades["open_date"]).dt.date)
        .groupby("open_day")["pair"]
        .apply(set)
        .to_dict()
    )
    
    all_days = sorted(set(bt_daily_pairs.keys()) | set(live_daily_pairs.keys()))
    
    print(f"\nDaily pair comparison ({len(all_days)} days):\n")
    daily_rows = []
    for day in all_days:
        bt_pairs = bt_daily_pairs.get(day, set())
        live_pairs_day = live_daily_pairs.get(day, set())
        common = bt_pairs & live_pairs_day
        bt_only = bt_pairs - live_pairs_day
        live_only = live_pairs_day - bt_pairs
        daily_rows.append({
            "Day": str(day),
            "BT Trades": len(bt_pairs),
            "Live Trades": len(live_pairs_day),
            "Common": len(common),
            "BT Only": len(bt_only),
            "Live Only": len(live_only),
            "BT Only Pairs": ", ".join(sorted(p.split("/")[0] for p in bt_only)) if bt_only else "-",
            "Live Only Pairs": ", ".join(sorted(p.split("/")[0] for p in live_only)) if live_only else "-",
        })
    
    daily_df = pd.DataFrame(daily_rows)
    display(daily_df)
    
    # Summary
    bt_all = set(backtest_trades["pair"].unique())
    live_all = set(live_trades["pair"].unique())
    print(f"\nOverall traded universe:")
    print(f"  Backtest unique pairs: {len(bt_all)}")
    print(f"  Live unique pairs:     {len(live_all)}")
    print(f"  Overlap:               {len(bt_all & live_all)}")
    if bt_all - live_all:
        print(f"  BT only:  {sorted(p.split('/')[0] for p in (bt_all - live_all))}")
    if live_all - bt_all:
        print(f"  Live only: {sorted(p.split('/')[0] for p in (live_all - bt_all))}")
elif BOT_MODE == "static":
    print("Pairlist is static — no dynamic pairlist analysis needed.")


## 2. Pair-by-Pair Comparison

Compare performance metrics for common pairs traded in both backtest and live.

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    fig = go.Figure()
    
    for idx, trade in backtest_trades.iterrows():
        fig.add_trace(go.Scatter(
            x=[trade['open_date'], trade['close_date']],
            y=[trade['pair'], trade['pair']],
            mode='lines+markers',
            name=f"BT: {trade['pair']}",
            line=dict(color='blue', width=3),
            marker=dict(size=8),
            showlegend=False,
            hovertemplate=f"<b>BACKTEST</b><br>Pair: {trade['pair']}<br>Open: %{{x}}<br>Profit: {trade['profit_ratio_pct']:.2f}%<extra></extra>"
        ))
    
    for idx, trade in live_trades.iterrows():
        fig.add_trace(go.Scatter(
            x=[trade['open_date'], trade['close_date']],
            y=[trade['pair'], trade['pair']],
            mode='lines+markers',
            name=f"Live: {trade['pair']}",
            line=dict(color='green', width=3, dash='dash'),
            marker=dict(size=8, symbol='diamond'),
            showlegend=False,
            hovertemplate=f"<b>LIVE</b><br>Pair: {trade['pair']}<br>Open: %{{x}}<br>Profit: {trade['profit_ratio_pct']:.2f}%<extra></extra>"
        ))
    
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='lines+markers',
        name='Backtest',
        line=dict(color='blue', width=3),
        marker=dict(size=8)
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='lines+markers',
        name='Live',
        line=dict(color='green', width=3, dash='dash'),
        marker=dict(size=8, symbol='diamond')
    ))
    
    fig.update_layout(
        title='Trade Timeline: Backtest vs Live',
        xaxis_title='Time',
        yaxis_title='Pair',
        height=600,
        hovermode='closest'
    )
    
    fig.show()
    
    print("\nTrade Timing Summary:")
    print(f"\nBacktest:")
    print(f"  First trade: {backtest_trades['open_date'].min()}")
    print(f"  Last trade: {backtest_trades['close_date'].max()}")
    print(f"  Total duration: {(backtest_trades['close_date'].max() - backtest_trades['open_date'].min()).days} days")
    
    print(f"\nLive:")
    print(f"  First trade: {live_trades['open_date'].min()}")
    print(f"  Last trade: {live_trades['close_date'].max()}")
    print(f"  Total duration: {(live_trades['close_date'].max() - live_trades['open_date'].min()).days} days")
else:
    print("No live trades for comparison")

## Pair Performance - Backtest

In [ ]:
if live_trades is not None:
    live_trades
else:
    print("No live trades loaded")

In [ ]:
backtest_pair_stats = backtest_trades.groupby('pair').agg({
    'profit_abs': ['sum', 'mean', 'count'],
    'profit_ratio': 'mean',
    'duration_hours': 'mean',
}).round(4)

backtest_pair_stats.columns = ['total_profit', 'avg_profit', 'trades', 'avg_profit_pct', 'avg_duration_h']
backtest_pair_stats['avg_profit_pct'] = (backtest_pair_stats['avg_profit_pct'] * 100).round(2)
backtest_pair_stats['win_rate'] = backtest_trades.groupby('pair').apply(
    lambda x: (len(x[x['profit_ratio'] > 0]) / len(x) * 100), include_groups=False
).round(2)
backtest_pair_stats = backtest_pair_stats.sort_values('total_profit', ascending=False)

print("\nTop 20 Pairs (Backtest):")
display(backtest_pair_stats.head(20))

## Pair Performance - Live

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    live_pair_stats = live_trades.groupby('pair').agg({
        'profit_abs': ['sum', 'mean', 'count'],
        'profit_ratio': 'mean',
        'duration_hours': 'mean',
    }).round(4)
    
    live_pair_stats.columns = ['total_profit', 'avg_profit', 'trades', 'avg_profit_pct', 'avg_duration_h']
    live_pair_stats['avg_profit_pct'] = (live_pair_stats['avg_profit_pct'] * 100).round(2)
    live_pair_stats['win_rate'] = live_trades.groupby('pair').apply(
        lambda x: (len(x[x['profit_ratio'] > 0]) / len(x) * 100), include_groups=False
    ).round(2)
    live_pair_stats = live_pair_stats.sort_values('total_profit', ascending=False)
    
    print("\nTop 20 Pairs (Live):")
    display(live_pair_stats.head(20))
else:
    print("No live trades available")

## Daily Profit - Backtest

In [ ]:
backtest_trades['close_date_only'] = pd.to_datetime(backtest_trades['close_date']).dt.date
backtest_daily = backtest_trades.groupby('close_date_only').agg({
    'profit_abs': ['sum', 'count']
}).reset_index()
backtest_daily.columns = ['date', 'profit', 'num_trades']
backtest_daily['cumulative'] = backtest_daily['profit'].cumsum()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily Profit (Backtest)', 'Daily Trade Count'),
    vertical_spacing=0.15
)

fig.add_trace(
    go.Bar(x=backtest_daily['date'], y=backtest_daily['profit'], name='Profit'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=backtest_daily['date'], y=backtest_daily['num_trades'], name='Trades', marker_color='orange'),
    row=2, col=1
)

fig.update_layout(height=600, showlegend=False)
fig.show()

## Daily Profit - Live

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    live_trades['close_date_only'] = pd.to_datetime(live_trades['close_date']).dt.date
    live_daily = live_trades.groupby('close_date_only').agg({
        'profit_abs': ['sum', 'count']
    }).reset_index()
    live_daily.columns = ['date', 'profit', 'num_trades']
    live_daily['cumulative'] = live_daily['profit'].cumsum()
    
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Daily Profit (Live)', 'Daily Trade Count'),
        vertical_spacing=0.15
    )
    
    fig.add_trace(
        go.Bar(x=live_daily['date'], y=live_daily['profit'], name='Profit', marker_color='green'),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(x=live_daily['date'], y=live_daily['num_trades'], name='Trades', marker_color='lightgreen'),
        row=2, col=1
    )
    
    fig.update_layout(height=600, showlegend=False)
    fig.show()
else:
    print("No live trades available")

## Exit Reasons Comparison

In [ ]:
backtest_exits = backtest_trades.groupby('exit_reason').agg({
    'profit_abs': ['sum', 'mean', 'count'],
    'profit_ratio': 'mean'
}).round(4)
backtest_exits.columns = ['total_profit', 'avg_profit', 'count', 'avg_profit_pct']
backtest_exits['avg_profit_pct'] = (backtest_exits['avg_profit_pct'] * 100).round(2)
backtest_exits = backtest_exits.sort_values('count', ascending=False)

print("\nExit Reasons (Backtest):")
display(backtest_exits)

fig = px.pie(backtest_exits, values='count', names=backtest_exits.index, title='Exit Reasons (Backtest)')
fig.update_layout(height=400)
fig.show()

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    live_exits = live_trades.groupby('exit_reason').agg({
        'profit_abs': ['sum', 'mean', 'count'],
        'profit_ratio': 'mean'
    }).round(4)
    live_exits.columns = ['total_profit', 'avg_profit', 'count', 'avg_profit_pct']
    live_exits['avg_profit_pct'] = (live_exits['avg_profit_pct'] * 100).round(2)
    live_exits = live_exits.sort_values('count', ascending=False)
    
    print("\nExit Reasons (Live):")
    display(live_exits)
    
    fig = px.pie(live_exits, values='count', names=live_exits.index, title='Exit Reasons (Live)')
    fig.update_layout(height=400)
    fig.show()
else:
    print("No live trades available")

## Profit Distribution

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Backtest Profit % Distribution', 'Live Profit % Distribution'),
)

fig.add_trace(
    go.Histogram(x=backtest_trades['profit_ratio_pct'], nbinsx=50, name='Backtest', marker_color='blue'),
    row=1, col=1
)

if live_trades is not None and len(live_trades) > 0:
    fig.add_trace(
        go.Histogram(x=live_trades['profit_ratio_pct'], nbinsx=50, name='Live', marker_color='green'),
        row=1, col=2
    )

fig.update_layout(height=400, showlegend=False)
fig.show()

## Best and Worst Trades - Backtest

In [ ]:
best_trades = backtest_trades.nlargest(10, 'profit_abs')[[
    'pair', 'open_date', 'close_date', 'profit_ratio_pct', 'profit_abs', 'duration_hours', 'exit_reason'
]]
print("Top 10 Best Trades (Backtest):")
display(best_trades)

worst_trades = backtest_trades.nsmallest(10, 'profit_abs')[[
    'pair', 'open_date', 'close_date', 'profit_ratio_pct', 'profit_abs', 'duration_hours', 'exit_reason'
]]
print("\nTop 10 Worst Trades (Backtest):")
display(worst_trades)

## Best and Worst Trades - Live

In [ ]:
if live_trades is not None and len(live_trades) > 0:
    best_trades_live = live_trades.nlargest(10, 'profit_abs')[[
        'pair', 'open_date', 'close_date', 'profit_ratio_pct', 'profit_abs', 'duration_hours', 'exit_reason'
    ]]
    print("Top 10 Best Trades (Live):")
    display(best_trades_live)
    
    worst_trades_live = live_trades.nsmallest(10, 'profit_abs')[[
        'pair', 'open_date', 'close_date', 'profit_ratio_pct', 'profit_abs', 'duration_hours', 'exit_reason'
    ]]
    print("\nTop 10 Worst Trades (Live):")
    display(worst_trades_live)
else:
    print("No live trades available")

## Visualize Trades for Specific Pair

Plot candlestick chart with trade entries and exits for a specific pair.

In [ ]:
def visualize_pair(pair, trades_df, data_processed, strategy_obj, live_trades_df=None):
    """Visualize trades for a specific pair with candlestick chart, overlaying live trades when provided."""
    if pair not in data_processed:
        print(f"No data available for {pair}")
        return

    df = data_processed[pair].copy()
    trades_for_pair = trades_df[trades_df["pair"] == pair].copy()
    
    live_trades = None
    if live_trades_df is not None:
        live_trades = live_trades_df[live_trades_df["pair"] == pair].copy()
        if live_trades.empty:
            live_trades = None
    
    if len(trades_for_pair) == 0 and live_trades is None:
        print(f"No trades found for {pair}")
        return

    plot_config = getattr(strategy_obj, "plot_config", {})
    
    trades_for_plot = trades_for_pair if len(trades_for_pair) > 0 else None
    
    fig = generate_candlestick_graph(
        pair=pair,
        data=df,
        trades=trades_for_plot,
        plot_config=plot_config,
    )

    if live_trades is not None and not live_trades.empty:
        live_trades["open_dt"] = pd.to_datetime(live_trades["open_date"])
        live_trades["close_dt"] = pd.to_datetime(live_trades["close_date"])

        fig.add_trace(
            go.Scatter(
                x=live_trades["open_dt"],
                y=live_trades["open_rate"],
                mode="markers",
                marker=dict(color="#00BFFF", size=9, symbol="diamond"),
                name="Live Entry",
                hovertemplate="Live entry<br>%{x}<br>Price: %{y}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=live_trades["close_dt"],
                y=live_trades["close_rate"],
                mode="markers",
                marker=dict(color="#FF8C00", size=9, symbol="square"),
                name="Live Exit",
                hovertemplate="Live exit<br>%{x}<br>Price: %{y}<extra></extra>",
            )
        )

    fig.update_layout(height=900, title=f"{pair} - Trades Visualization")
    fig.show()

    if len(trades_for_pair) > 0:
        print(f"\nBacktest Trade Statistics for {pair}:")
        print(f"  Total Trades: {len(trades_for_pair)}")
        print(f"  Win Rate: {(len(trades_for_pair[trades_for_pair['profit_ratio'] > 0]) / len(trades_for_pair) * 100):.2f}%")
        print(f"  Total Profit: {trades_for_pair['profit_abs'].sum():.2f}")
        print(f"  Avg Profit: {trades_for_pair['profit_ratio'].mean() * 100:.2f}%")
    
    if live_trades is not None and not live_trades.empty:
        print(f"\nLive Trade Statistics for {pair}:")
        print(f"  Total Trades: {len(live_trades)}")
        if 'profit_ratio' in live_trades.columns:
            print(f"  Win Rate: {(len(live_trades[live_trades['profit_ratio'] > 0]) / len(live_trades) * 100):.2f}%")
        if 'profit_abs' in live_trades.columns:
            print(f"  Total Profit: {live_trades['profit_abs'].sum():.2f}")
        if 'profit_ratio' in live_trades.columns:
            print(f"  Avg Profit: {live_trades['profit_ratio'].mean() * 100:.2f}%")

    return trades_for_pair if len(trades_for_pair) > 0 else live_trades

In [ ]:
import pandas as pd

def visualize_pair_range(
    pair,
    trades_df,
    data_processed,
    strategy_obj,
    live_trades_df=None,
    start_date=None,
    end_date=None,
):
    """Visualize pair with optional date range filtering."""
    df = data_processed[pair].copy()
    trades_for_pair = trades_df[trades_df["pair"] == pair].copy()

    live_trades_local = None
    if live_trades_df is not None:
        live_trades_local = live_trades_df[live_trades_df["pair"] == pair].copy()
        if live_trades_local.empty:
            live_trades_local = None

    start_ts = pd.to_datetime(start_date, utc=True) if start_date else None
    end_ts = pd.to_datetime(end_date, utc=True) if end_date else None

    def _overlaps_window(open_s, close_s):
        mask = pd.Series(True, index=open_s.index)
        if start_ts is not None:
            mask &= (close_s.isna()) | (close_s >= start_ts)
        if end_ts is not None:
            mask &= open_s <= end_ts
        return mask

    if start_ts is not None or end_ts is not None:
        df["date"] = pd.to_datetime(df["date"], utc=True)
        m = pd.Series(True, index=df.index)
        if start_ts is not None:
            m &= df["date"] >= start_ts
        if end_ts is not None:
            m &= df["date"] <= end_ts
        df = df.loc[m].copy()

    if start_ts is not None or end_ts is not None:
        bt_open = pd.to_datetime(trades_for_pair["open_date"], utc=True)
        bt_close = pd.to_datetime(trades_for_pair["close_date"], utc=True)
        trades_for_pair = trades_for_pair.loc[_overlaps_window(bt_open, bt_close)].copy()

        if live_trades_local is not None:
            lv_open = pd.to_datetime(live_trades_local["open_date"], utc=True)
            lv_close = pd.to_datetime(live_trades_local["close_date"], utc=True, errors="coerce")
            live_trades_local = live_trades_local.loc[_overlaps_window(lv_open, lv_close)].copy()
            if live_trades_local.empty:
                live_trades_local = None

In [ ]:
processed

In [ ]:
backtest_trades.tail()

In [ ]:
# Visualize top performing pair from backtest
if len(backtest_pair_stats) > 0:
    top_pair = backtest_pair_stats.index[0]
    print(f"Visualizing top performing pair: {top_pair}\n")
    visualize_pair(top_pair, backtest_trades, processed, strategy, live_trades_df=live_trades)
else:
    print("No pairs to visualize")

In [ ]:
# Optional: Visualize a custom pair
CUSTOM_PAIR = "BTC/USDC:USDC"  # Change this to any pair you want to visualize

if CUSTOM_PAIR in backtest_trades['pair'].values:
    print(f"\nVisualizing custom pair: {CUSTOM_PAIR}\n")
    visualize_pair(CUSTOM_PAIR, backtest_trades, processed, strategy)
else:
    print(f"Pair {CUSTOM_PAIR} not found in backtest results")